# Reranker — Final Evaluation

One consolidated, GPU-free analysis of a trained kernel reranker. It answers, over
a held-out set of KernelBench runs:

1. **Dataset summary** — what the compiled candidate pool looks like (§1).
2. **Ranking metrics** — is the top pick *correct*, and does the reranker find the
   *fastest correct* kernel per problem (§2)?
3. **Reranker vs random vs oracle** — is the reranker worth it, for speed and for
   correctness (§3)?
4. **Score distributions** — do the scores separate the outcomes we care about,
   given the scores are only calibrated *within* a problem (§4)?
5. **Robustness & export** — bootstrap CIs, per-slice breakdowns, `metrics.json` (§5).

**Two-stage inputs (no GPU needed here):**
- `EVAL_TABLE` — reranker-independent `eval_table.jsonl` from
  `reranker.src.eval_pipeline.build_eval_table` (outcomes + kernel/baseline timings).
- `RERANKERS` — one `scores/<name>.jsonl` per reranker from
  `reranker.src.eval_pipeline.score_run`, joined on `(run_name, kernel_file)`.

Add more entries to `RERANKERS` to compare rerankers **without** rebuilding the eval
table. All candidate pools are **compiled kernels only** — deployment compile-checks
candidates for free before the reranker runs.

In [ ]:
# === Parameters ==============================================================
EVAL_TABLE = "reranker/data/eval/eval_table.jsonl"          # Stage 1a artifact
RERANKERS = {                                               # name -> Stage 1b scores/<name>.jsonl
    "listwise_l56": "reranker/data/eval/scores/listwise_l56.jsonl",
}

# Ground-truth speedup statistic. "mean" = KernelBench convention (baseline_mean /
# runtime_mean); "min" = noise-robust min/min (matches the l56 listwise labels).
# Flip freely — the scores artifact is unaffected, so no re-scoring is needed.
SPEEDUP_STAT = "mean"           # "mean" | "min"

POOL_RUNS = True                # a "problem" = (level, problem_id) pooled across runs
TAUS = [0.5, 1.0, 1.5, 2.0]     # speedup thresholds (dataset property + fast@k)
SPEEDUP_CAP = 4.0               # cap for mean-speedup aggregates (heavy tails)
K_MAX = 16                      # fast@k / recall@k curves
N_RANDOM_DRAWS = 200            # random-selection simulations (§3), averaged
N_BOOTSTRAP = 1000              # bootstrap resamples over problems (§5)
FASTP_EDGES = [0.5, 0.8, 1, 1.5, 2, 3, 5, 10]   # speedup buckets (§1 property, §4)

# Graded-relevance mapping for NDCG — MUST match the training listwise config so
# the NDCG here is comparable to the MLflow training curve (l56: lo=0.2, hi=4.0,
# quant=0.1). rel = 1 + speed_p(speedup) for correct kernels, else 0.
SPEEDUP_LO, SPEEDUP_HI, SPEED_QUANT = 0.2, 4.0, 0.1

OUT_DIR = "reranker/data/eval/report"   # CSVs, plots and metrics.json land here
SEED = 42

In [ ]:
import json
import os
from math import comb
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import seaborn as sns
    sns.set_theme(style="whitegrid")
except Exception:
    plt.style.use("default")

os.makedirs(OUT_DIR, exist_ok=True)
rng = np.random.default_rng(SEED)

# Fixed categorical colors (reranker blue, random amber, oracle green) + inks.
C_RANDOM, C_ORACLE = "#eda100", "#1baf7a"
INK, MUTED, BASELINE = "#0b0b0b", "#898781", "#c3c2b7"
RERANKER_PALETTE = ["#2a78d6", "#7b3fb5", "#d94a8c", "#00968f", "#c86b1f"]

GROUP_KEYS = ["level", "problem_id"] if POOL_RUNS else ["run_name", "level", "problem_id"]


def speed_p(speedup, lo=None, hi=None, quant=None):
    # Map an absolute speedup to p in [0,1] on a log2 scale (mirrors
    # reranker/src/data/labels.py::speed_p). lo->0, hi->1, clamped, snapped to quant.
    lo = SPEEDUP_LO if lo is None else lo
    hi = SPEEDUP_HI if hi is None else hi
    quant = SPEED_QUANT if quant is None else quant
    if speedup is None or speedup <= 0:
        return 0.0
    lg, llo, lhi = np.log2(speedup), np.log2(lo), np.log2(hi)
    p = min(1.0, max(0.0, (lg - llo) / (lhi - llo)))
    if quant > 0:
        p = min(1.0, max(0.0, round(p / quant) * quant))
    return float(p)


def graded_ndcg(scores, rels):
    # NDCG of the score-induced order vs graded relevance (gain=2^rel-1,
    # 1/log2(rank+2) discount) - mirrors listwise/trainer.py::_graded_ndcg.
    scores = np.asarray(scores, float); rels = np.asarray(rels, float)
    n = len(scores)
    if n < 1:
        return 0.0
    gains = 2.0 ** rels - 1.0
    disc = 1.0 / np.log2(np.arange(n) + 2.0)
    idcg = (np.sort(gains)[::-1] * disc).sum()
    if idcg <= 0:
        return 0.0
    dcg = (gains[np.argsort(-scores, kind="stable")] * disc).sum()
    return float(dcg / idcg)


def auc_rank(pos_scores, neg_scores):
    # AUC via Mann-Whitney U (tie-averaged ranks; no sklearn). None if a class is empty.
    npos, nneg = len(pos_scores), len(neg_scores)
    if npos == 0 or nneg == 0:
        return None
    alls = np.concatenate([pos_scores, neg_scores])
    ranks = pd.Series(alls).rank(method="average").to_numpy()
    r_pos = ranks[:npos].sum()
    return float((r_pos - npos * (npos + 1) / 2) / (npos * nneg))


def cap(v):
    return np.minimum(np.asarray(v, float), SPEEDUP_CAP)

In [ ]:
# --- Load eval table + reranker scores; compute per-kernel speedup ----------
ev = pd.read_json(EVAL_TABLE, lines=True)
print(f"eval table: {len(ev)} kernels | runs={ev['run_name'].nunique()} | "
      f"levels={sorted(ev['level'].unique())} | problems(pooled)="
      f"{ev.groupby(GROUP_KEYS).ngroups}")

rt = ev[f"runtime_{SPEEDUP_STAT}"]
bl = ev[f"baseline_{SPEEDUP_STAT}"]
gradable = ev["correct"] & rt.notna() & (rt > 0) & bl.notna() & (bl > 0)
ev["has_baseline"] = bl.notna() & (bl > 0)
ev["gradable"] = gradable
# speedup: correct & gradable -> baseline/runtime ; else 0 (a wrong or ungradable
# pick buys no measured speedup). Correct-but-ungradable is tracked separately.
ev["speedup"] = np.where(gradable, bl / rt.where(rt > 0, np.nan), 0.0)
ev["ungraded_correct"] = ev["correct"] & ~ev["gradable"]
print(f"SPEEDUP_STAT={SPEEDUP_STAT}: {int(gradable.sum())} gradable correct kernels, "
      f"{int(ev['ungraded_correct'].sum())} correct-but-ungradable (no baseline/runtime)")

# Join each reranker's scores on (run_name, kernel_file).
scored = {}
for name, path in RERANKERS.items():
    s = pd.read_json(path, lines=True)
    need = {"run_name", "kernel_file", "score_logit", "score_sigmoid"}
    missing = need - set(s.columns)
    if missing:
        raise ValueError(f"scores file {path} missing columns {missing}")
    df = ev.merge(s[["run_name", "kernel_file", "score_logit", "score_sigmoid",
                     "truncated"]],
                  on=["run_name", "kernel_file"], how="left")
    n_scored = df["score_logit"].notna().sum()
    n_comp = int(df["compiled"].sum())
    print(f"[{name}] joined {n_scored}/{len(df)} scored "
          f"({df.loc[df['compiled'], 'score_logit'].notna().sum()}/{n_comp} of compiled)")
    scored[name] = df

RNAMES = list(scored.keys())
RCOLORS = {n: RERANKER_PALETTE[i % len(RERANKER_PALETTE)] for i, n in enumerate(RNAMES)}

## §1 — Dataset summary (compiled pool)

Reranker-independent (from the eval table alone). One compile-rate line, then everything
over the **compiled** pool the reranker actually ranks.

In [ ]:
# --- §1 headline: compile rate + funnel over the compiled pool --------------
n_total = len(ev)
n_comp = int(ev["compiled"].sum())
comp = ev[ev["compiled"]].copy()
n_correct = int(comp["correct"].sum())
n_gradable = int(comp["gradable"].sum())
n_problems = comp.groupby(GROUP_KEYS).ngroups

print(f"total kernels        : {n_total}")
print(f"compiled             : {n_comp}  ({n_comp/max(n_total,1):.1%} of all)  <- the reranker's pool")
print(f"  of compiled -> correct        : {n_correct}  ({n_correct/max(n_comp,1):.1%})")
print(f"  of compiled -> correct+gradable: {n_gradable}  ({n_gradable/max(n_comp,1):.1%})")
print(f"problems (compiled)  : {n_problems}")
sizes = comp.groupby(GROUP_KEYS).size()
print(f"compiled kernels/problem: mean={sizes.mean():.1f}  min={sizes.min()}  max={sizes.max()}")

# Speedup aggregates over the compiled pool (SPEEDUP_STAT).
su_all = comp["speedup"].to_numpy()                 # incorrect/ungradable -> 0
su_cor = comp.loc[comp["gradable"], "speedup"].to_numpy()
def _agg(v):
    v = np.asarray(v, float)
    if len(v) == 0:
        return dict(mean=np.nan, mean_capped=np.nan, median=np.nan, min=np.nan, max=np.nan)
    return dict(mean=v.mean(), mean_capped=cap(v).mean(), median=float(np.median(v)),
                min=float(v.min()), max=float(v.max()))
agg = pd.DataFrame({"all compiled (incorrect=0)": _agg(su_all),
                    "correct+gradable only": _agg(su_cor)}).T
print("\nspeedup aggregates:")
print(agg.round(3).to_string())

In [ ]:
# --- §1 per-problem table (-> CSV) + #correct histogram + speed ceiling ------
def _per_problem(g):
    su = g["speedup"].to_numpy()
    d = {"n_compiled": len(g), "n_correct": int(g["correct"].sum()),
         "best_speedup": float(su.max()) if len(su) else 0.0,
         "avg_speedup": float(su.mean()) if len(su) else 0.0}
    for t in TAUS:
        d[f"n_speedup>{t}"] = int((su > t).sum())
    return pd.Series(d)

pp = comp.groupby(GROUP_KEYS).apply(_per_problem, include_groups=False).reset_index()
pp_path = os.path.join(OUT_DIR, "per_problem_summary.csv")
pp.to_csv(pp_path, index=False)
print(f"per-problem table -> {pp_path}  ({len(pp)} problems)")

n_unsolvable = int((pp["n_correct"] == 0).sum())
print(f"unsolvable problems (0 correct): {n_unsolvable}/{len(pp)} "
      f"({n_unsolvable/max(len(pp),1):.1%})  <- reranker's hard ceiling")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.hist(pp["n_correct"], bins=range(0, int(pp["n_correct"].max()) + 2),
         color="#2a78d6", alpha=0.85, align="left")
ax1.set_xlabel("# correct kernels in problem"); ax1.set_ylabel("# problems")
ax1.set_title("Correct-per-problem (compiled pool)")

# Dataset speed ceiling: fraction of problems whose ORACLE clears each tau.
oracle_su = pp["best_speedup"].to_numpy()
ceil = [ (oracle_su >= t).mean() for t in FASTP_EDGES ]
ax2.bar([str(t) for t in FASTP_EDGES], ceil, color="#1baf7a", alpha=0.85)
for i, v in enumerate(ceil):
    ax2.text(i, v, f"{v:.2f}", ha="center", va="bottom", fontsize=8)
ax2.set_xlabel(f"speedup threshold ({SPEEDUP_STAT})"); ax2.set_ylabel("frac of problems w/ oracle >= tau")
ax2.set_title("Dataset speed ceiling (property, not a reranker metric)")
ax2.set_ylim(0, 1.02)
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "sec1_summary.png"), dpi=140); plt.show()

## §2 — Ranking metrics: correct at the top, and finding the *fastest correct*

Per problem over the compiled pool, averaged across problems, per reranker. Absolute
`fast@1(τ)` is **not** here (it tracks dataset difficulty, not the reranker — see §1's
speed ceiling); the speed metrics below are **relative** and computed on *solvable*
problems so difficulty is factored out.

| metric | meaning |
|---|---|
| `correct@1 (all / solvable)` | top pick is correct, over all / over solvable problems |
| `speed_attainment@1` | picked speedup / oracle speedup ∈ [0,1] — how much of the best speed we capture |
| `found_fastest@1` | pick **is** the single fastest correct kernel (exact-best hit rate) |
| `truefastest_top1 / top3` | the actual fastest kernel is ranked #1 / within top-3 |
| `ndcg_graded` | NDCG of the ranking vs `rel=1+speed_p(speedup)` (comparable to training) |
| `correctness_pair_acc` | correct-vs-wrong pairs ordered right (the gate) |
| `speed_pair_acc` | fast-vs-slow pairs among correct kernels ordered right |

In [ ]:
# --- §2 build per-problem groups (per reranker) and compute ranking metrics --
def build_groups(df):
    # Per-problem arrays over the compiled + scored pool.
    pool = df[df["compiled"] & df["score_logit"].notna()]
    groups = []
    for keys, g in pool.groupby(GROUP_KEYS):
        su = g["speedup"].to_numpy(float)
        cor = g["correct"].to_numpy(bool)
        sc = g["score_logit"].to_numpy(float)
        rel = np.array([1.0 + speed_p(s) if (c and s > 0) else 0.0
                        for c, s in zip(cor, su)])
        groups.append(dict(keys=keys, su=su, cor=cor, sc=sc, rel=rel))
    return groups


def ranking_metrics(groups):
    n = len(groups)
    picked_correct_all, picked_correct_solv = [], []
    attain, found_fastest, tf_top1, tf_top3 = [], [], [], []
    ndcgs = []
    corr_c = corr_t = sp_c = sp_t = 0
    for g in groups:
        su, cor, sc, rel = g["su"], g["cor"], g["sc"], g["rel"]
        pick = int(np.argmax(sc))
        picked_correct_all.append(bool(cor[pick]))
        if cor.any():
            picked_correct_solv.append(bool(cor[pick]))
        oracle = su.max()
        if oracle > 0:                                    # speed-solvable
            attain.append(su[pick] / oracle)
            found_fastest.append(bool(np.isclose(su[pick], oracle)))
            # rank of the true-fastest kernel by score (0 = top)
            fastest_idx = int(np.argmax(su))
            rank_of = int((sc > sc[fastest_idx]).sum())   # #kernels scored strictly higher
            tf_top1.append(rank_of == 0)
            tf_top3.append(rank_of < 3)
        if (rel > 0).any():
            ndcgs.append(graded_ndcg(sc, rel))
        # pairwise accuracies within the problem
        more = rel[:, None] > rel[None, :]
        better = (sc[:, None] - sc[None, :]) > 0
        j_wrong = rel[None, :] == 0
        corr_mask = more & j_wrong
        sp_mask = more & ~j_wrong
        corr_c += int((corr_mask & better).sum()); corr_t += int(corr_mask.sum())
        sp_c += int((sp_mask & better).sum()); sp_t += int(sp_mask.sum())
    return {
        "n_problems": n,
        "correct@1 (all)": np.mean(picked_correct_all) if picked_correct_all else np.nan,
        "correct@1 (solvable)": np.mean(picked_correct_solv) if picked_correct_solv else np.nan,
        "speed_attainment@1": np.mean(attain) if attain else np.nan,
        "found_fastest@1": np.mean(found_fastest) if found_fastest else np.nan,
        "truefastest_top1": np.mean(tf_top1) if tf_top1 else np.nan,
        "truefastest_top3": np.mean(tf_top3) if tf_top3 else np.nan,
        "ndcg_graded": np.mean(ndcgs) if ndcgs else np.nan,
        "correctness_pair_acc": corr_c / max(corr_t, 1),
        "speed_pair_acc": sp_c / max(sp_t, 1),
    }

groups_by = {name: build_groups(df) for name, df in scored.items()}
sec2 = pd.DataFrame({name: ranking_metrics(groups_by[name]) for name in RNAMES})
print(sec2.round(3).to_string())
sec2.to_csv(os.path.join(OUT_DIR, "sec2_ranking_metrics.csv"))

In [ ]:
# --- §2 plot: the two jobs side by side, per reranker ------------------------
show = ["correct@1 (solvable)", "speed_attainment@1", "found_fastest@1",
        "ndcg_graded", "correctness_pair_acc", "speed_pair_acc"]
x = np.arange(len(show)); w = 0.8 / max(len(RNAMES), 1)
fig, ax = plt.subplots(figsize=(12, 4.5))
for i, name in enumerate(RNAMES):
    vals = [sec2.loc[m, name] for m in show]
    ax.bar(x + (i - (len(RNAMES) - 1) / 2) * w, vals, width=w * 0.95,
           color=RCOLORS[name], label=name)
ax.axhline(0.5, color=BASELINE, ls=":", lw=1)
ax.set_xticks(x); ax.set_xticklabels(show, rotation=20, ha="right")
ax.set_ylim(0, 1.02); ax.set_ylabel("score")
ax.set_title("§2 Ranking metrics (compiled pool, solvable where relative)")
ax.legend(frameon=False, fontsize=9)
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "sec2_ranking.png"), dpi=140); plt.show()

## §3 — Reranker vs random vs oracle

Candidate pool = compiled kernels. **Random is simulated `N_RANDOM_DRAWS` times** (uniform
pick per problem) and averaged; the whisker is its 5–95% band. Oracle and random are
reranker-independent, so they appear once; each reranker gets its own bar.

In [ ]:
# --- §3 per-problem policy values (uses the pooled compiled scored pool) -----
# Reranker/oracle need scores; random/oracle are reranker-independent. Use the
# union pool = compiled kernels that ALL rerankers scored, so comparisons align.
base_groups = groups_by[RNAMES[0]]   # same problem set & su/cor across rerankers
# sanity: identical problem keys across rerankers
for name in RNAMES[1:]:
    assert [g["keys"] for g in groups_by[name]] == [g["keys"] for g in base_groups], \
        "reranker pools differ; ensure all scored the same eval table"

su_list = [g["su"] for g in base_groups]
cor_list = [g["cor"] for g in base_groups]
n_prob = len(base_groups)
n_solvable_correct = int(sum(c.any() for c in cor_list))          # oracle correctness ceiling
n_solvable_speed = int(sum(su.max() > 0 for su in su_list))

# Reranker picks per reranker
rer_speed = {name: np.array([g["su"][int(np.argmax(g["sc"]))] for g in groups_by[name]])
             for name in RNAMES}
rer_correct = {name: int(sum(g["cor"][int(np.argmax(g["sc"]))] for g in groups_by[name]))
               for name in RNAMES}
oracle_speed = np.array([su.max() for su in su_list])

# Simulated random: R draws, each picks one candidate/problem uniformly.
rnd_speed_means, rnd_ncorrect = [], []
for t in range(N_RANDOM_DRAWS):
    r = np.random.default_rng(SEED + t)
    idx = [r.integers(len(su)) for su in su_list]
    rnd_speed_means.append(cap([su_list[i][j] for i, j in enumerate(idx)]).mean())
    rnd_ncorrect.append(int(sum(cor_list[i][j] for i, j in enumerate(idx))))
rnd_speed_means = np.array(rnd_speed_means); rnd_ncorrect = np.array(rnd_ncorrect)
rnd_lo, rnd_hi = np.percentile(rnd_speed_means, [5, 95])
rnc_lo, rnc_hi = np.percentile(rnd_ncorrect, [5, 95])

print(f"{n_prob} problems | solvable(correct)={n_solvable_correct} | "
      f"solvable(speed)={n_solvable_speed}")
for name in RNAMES:
    print(f"[{name}] mean capped speedup={cap(rer_speed[name]).mean():.3f}  "
          f"#correct@1={rer_correct[name]}")
print(f"[random] mean capped speedup={rnd_speed_means.mean():.3f} "
      f"[{rnd_lo:.3f},{rnd_hi:.3f}]  #correct@1={rnd_ncorrect.mean():.1f}")
print(f"[oracle] mean capped speedup={cap(oracle_speed).mean():.3f}  "
      f"#correct(ceiling)={n_solvable_correct}")

In [ ]:
# --- §3 Plot 1 (speed) and Plot 2 (correctness) — two separate plots ---------
fig, ax = plt.subplots(figsize=(9, 4.6))
labels = list(RNAMES) + ["random\n(avg)", "oracle"]
vals = [cap(rer_speed[n]).mean() for n in RNAMES] + [rnd_speed_means.mean(),
                                                     cap(oracle_speed).mean()]
colors = [RCOLORS[n] for n in RNAMES] + [C_RANDOM, C_ORACLE]
bars = ax.bar(labels, vals, color=colors, width=0.6)
# 5-95% band on the random bar
ri = len(RNAMES); mid = (rnd_lo + rnd_hi) / 2
ax.errorbar([ri], [mid], yerr=[[mid - rnd_lo], [rnd_hi - mid]], fmt="none",
            ecolor=INK, elinewidth=1.4, capsize=6)
for i, v in enumerate(vals):
    ax.text(i, v, f" {v:.2f}", ha="center", va="bottom", fontsize=9)
ax.axhline(1.0, color=BASELINE, ls=":", lw=1)
ax.set_ylabel(f"mean picked speedup ({SPEEDUP_STAT}, capped {SPEEDUP_CAP:g})")
ax.set_title(f"§3 Plot 1 — Speed of the pick ({n_prob} problems)")
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "sec3_speed.png"), dpi=140); plt.show()

fig, ax = plt.subplots(figsize=(9, 4.6))
vals_c = [rer_correct[n] for n in RNAMES] + [rnd_ncorrect.mean(), n_solvable_correct]
bars = ax.bar(labels, vals_c, color=colors, width=0.6)
ax.errorbar([ri], [rnd_ncorrect.mean()],
            yerr=[[rnd_ncorrect.mean() - rnc_lo], [rnc_hi - rnd_ncorrect.mean()]],
            fmt="none", ecolor=INK, elinewidth=1.4, capsize=6)
for i, v in enumerate(vals_c):
    ax.text(i, v, f" {v:.0f}", ha="center", va="bottom", fontsize=9)
ax.set_ylabel("# problems with a correct pick")
ax.set_title(f"§3 Plot 2 — Correctness of the pick (oracle ceiling = {n_solvable_correct} solvable)")
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "sec3_correct.png"), dpi=140); plt.show()

In [ ]:
# --- §3 fast@k / recall@k ----------------------------------------------------
def random_at_k(su_all, tau, k):
    n = len(su_all); m = int((su_all >= tau).sum()); kk = min(k, n)
    if m == 0: return 0.0
    if n - m < kk: return 1.0
    return 1.0 - comb(n - m, kk) / comb(n, kk)

ks = list(range(1, K_MAX + 1))
fig, axes = plt.subplots(1, len(TAUS[1:]), figsize=(5 * len(TAUS[1:]), 4.4),
                         sharey=True, squeeze=False)
for ax, tau in zip(axes[0], TAUS[1:]):   # skip 0.5 for the @k view; keep >=1 thresholds
    ceiling = np.mean([1.0 if (su >= tau).any() else 0.0 for su in su_list])
    rnd = [np.mean([random_at_k(su, tau, k) for su in su_list]) for k in ks]
    ax.plot(ks, rnd, "--s", color=MUTED, lw=1.4, ms=3, label="random@k")
    ax.axhline(ceiling, color=C_ORACLE, ls=":", lw=1.4, label=f"ceiling ({ceiling:.2f})")
    for name in RNAMES:
        sortd = [g["su"][np.argsort(-g["sc"], kind="stable")] for g in groups_by[name]]
        rer = [np.mean([1.0 if (s[:k] >= tau).any() else 0.0 for s in sortd]) for k in ks]
        ax.plot(ks, rer, "-o", color=RCOLORS[name], lw=2, ms=3, label=f"{name} top-k")
    ax.set_title(f"speedup >= {tau}"); ax.set_xlabel("k (top-k by score)")
    ax.set_xticks(ks[::2]); ax.set_ylim(0, 1.02); ax.legend(fontsize=8, loc="lower right")
axes[0][0].set_ylabel("frac problems with a qualifying kernel in top-k")
fig.suptitle(f"§3 fast@k / recall@k — {n_prob} problems", y=1.02)
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "sec3_fast_at_k.png"), dpi=140); plt.show()

In [ ]:
# --- §3 regret CDF (oracle - reranker picked speedup, capped) ----------------
fig, ax = plt.subplots(figsize=(8, 4.4))
solv_mask = oracle_speed > 0
for name in RNAMES:
    regret = np.sort(cap(oracle_speed[solv_mask]) - cap(rer_speed[name][solv_mask]))[::-1]
    frac = np.arange(1, len(regret) + 1) / len(regret)
    ax.plot(frac, regret, color=RCOLORS[name], lw=1.8, label=f"{name} (mean {regret.mean():.3f})")
ax.axhline(0, color=BASELINE, lw=1)
ax.set_xlabel("speed-solvable problems (sorted by regret, desc)")
ax.set_ylabel(f"oracle − reranker speedup (capped {SPEEDUP_CAP:g})")
ax.set_title("§3 Capped speed regret"); ax.legend(frameon=False, fontsize=9)
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "sec3_regret.png"), dpi=140); plt.show()

## §4 — Score distributions (within-problem)

**Caveat:** the LambdaRank reranker is trained to rank *within* a problem, so its raw scores
are calibrated within-problem, **not** globally — a naive global histogram can overlap even
when every within-problem ranking is perfect. So we use shift-invariant, within-problem views:
per-problem **rank percentile** by outcome, a **speed-discrimination** view over correct kernels,
and per-problem **AUC**. One set of plots per reranker.

In [ ]:
# --- §4 per reranker: rank-percentile by outcome, speed-discrimination, AUC --
def outcome_class(correct, speedup):
    if not correct:
        return "compiled-wrong"
    return "correct-fast(>=1x)" if speedup >= 1.0 else "correct-slow(<1x)"

CLASS_ORDER = ["compiled-wrong", "correct-slow(<1x)", "correct-fast(>=1x)"]
CLASS_COLOR = {"compiled-wrong": "#c0504d", "correct-slow(<1x)": "#eda100",
               "correct-fast(>=1x)": "#1baf7a"}

for name in RNAMES:
    df = scored[name]
    pool = df[df["compiled"] & df["score_logit"].notna()].copy()
    # within-problem rank percentile of the score (1 = top within its problem)
    pool["pct"] = pool.groupby(GROUP_KEYS)["score_logit"].rank(pct=True)
    pool["cls"] = [outcome_class(c, s) for c, s in zip(pool["correct"], pool["speedup"])]

    # per-problem AUC (correct vs compiled-wrong) and Spearman(score, speedup|correct)
    aucs, spears = [], []
    for _, g in pool.groupby(GROUP_KEYS):
        pos = g.loc[g["correct"], "score_logit"].to_numpy()
        neg = g.loc[~g["correct"], "score_logit"].to_numpy()
        a = auc_rank(pos, neg)
        if a is not None:
            aucs.append(a)
        cg = g[g["correct"] & (g["speedup"] > 0)]
        if len(cg) >= 3 and cg["speedup"].nunique() >= 2:
            rs = cg["score_logit"].rank(); rp = cg["speedup"].rank()
            if rs.std() > 0 and rp.std() > 0:
                spears.append(float(np.corrcoef(rs, rp)[0, 1]))

    fig, axes = plt.subplots(1, 3, figsize=(16, 4.4))
    # (a) rank percentile by outcome
    for cls in CLASS_ORDER:
        v = pool.loc[pool["cls"] == cls, "pct"].to_numpy()
        if len(v):
            axes[0].hist(v, bins=20, range=(0, 1), alpha=0.55, density=True,
                         color=CLASS_COLOR[cls], label=f"{cls} (n={len(v)})")
    axes[0].set_xlabel("within-problem score percentile"); axes[0].set_ylabel("density")
    axes[0].set_title(f"[{name}] rank percentile by outcome"); axes[0].legend(fontsize=8)

    # (b) speed-discrimination: correct kernels bucketed by speedup vs rank pct
    cor = pool[pool["correct"] & (pool["speedup"] > 0)].copy()
    if len(cor):
        cor["bucket"] = pd.cut(cor["speedup"], bins=[0] + FASTP_EDGES, right=False)
        data, labs = [], []
        for b, gg in cor.groupby("bucket", observed=True):
            if len(gg) >= 3:
                data.append(gg["pct"].to_numpy()); labs.append(str(b))
        if data:
            axes[1].boxplot(data, showfliers=False)
            axes[1].set_xticks(range(1, len(labs) + 1))
            axes[1].set_xticklabels(labs, rotation=30, ha="right", fontsize=7)
    axes[1].set_ylabel("within-problem score percentile")
    sp_mean = np.mean(spears) if spears else float("nan")
    axes[1].set_title(f"[{name}] speed-discrimination (correct)\n"
                      f"mean within-problem Spearman(score,speedup)={sp_mean:.3f}")

    # (c) per-problem AUC (correct vs wrong)
    if aucs:
        axes[2].hist(aucs, bins=20, range=(0, 1), color="#2a78d6", alpha=0.85)
        axes[2].axvline(0.5, color=BASELINE, ls=":", lw=1)
        axes[2].axvline(np.mean(aucs), color=INK, lw=1.5,
                        label=f"mean {np.mean(aucs):.3f}")
        axes[2].legend(fontsize=8)
    axes[2].set_xlabel("per-problem AUC (correct vs compiled-wrong)")
    axes[2].set_ylabel("# problems"); axes[2].set_title(f"[{name}] within-problem AUC")
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, f"sec4_{name}.png"), dpi=140); plt.show()

## §5 — Robustness & export

Bootstrap 95% CIs over problems for the headline metrics, per-level / per-run breakdowns,
and a `metrics.json` capturing every headline number + parameters for cross-checkpoint
comparison.

In [ ]:
# --- §5 bootstrap CIs over problems -----------------------------------------
def headline(groups):
    m = ranking_metrics(groups)
    rer = np.array([g["su"][int(np.argmax(g["sc"]))] for g in groups])
    m["mean_picked_speedup_capped"] = float(cap(rer).mean())
    return m

BOOT_KEYS = ["correct@1 (solvable)", "speed_attainment@1", "found_fastest@1",
             "ndcg_graded", "mean_picked_speedup_capped"]
sec5_ci = {}
for name in RNAMES:
    gs = groups_by[name]; n = len(gs)
    samples = {k: [] for k in BOOT_KEYS}
    for b in range(N_BOOTSTRAP):
        r = np.random.default_rng(SEED + b)
        idx = r.integers(0, n, n)
        hb = headline([gs[i] for i in idx])
        for k in BOOT_KEYS:
            samples[k].append(hb[k])
    base = headline(gs)
    sec5_ci[name] = {k: (base[k], float(np.nanpercentile(samples[k], 2.5)),
                         float(np.nanpercentile(samples[k], 97.5))) for k in BOOT_KEYS}

print("bootstrap 95% CI (point [lo, hi]):")
for name in RNAMES:
    print(f"[{name}]")
    for k in BOOT_KEYS:
        p, lo, hi = sec5_ci[name][k]
        print(f"    {k:32s} {p:.3f}  [{lo:.3f}, {hi:.3f}]")

In [ ]:
# --- §5 per-level / per-run breakdown ---------------------------------------
def breakdown(df, by):
    rows = []
    pool = df[df["compiled"] & df["score_logit"].notna()]
    for key, g in pool.groupby(by):
        gg = g.copy()
        groups = []
        for _, gp in gg.groupby(GROUP_KEYS):
            groups.append(dict(su=gp["speedup"].to_numpy(float),
                               cor=gp["correct"].to_numpy(bool),
                               sc=gp["score_logit"].to_numpy(float),
                               rel=np.array([1.0 + speed_p(s) if (c and s > 0) else 0.0
                                             for c, s in zip(gp["correct"], gp["speedup"])])))
        m = ranking_metrics(groups)
        rows.append({by: key, "n_problems": m["n_problems"],
                     "correct@1(solv)": m["correct@1 (solvable)"],
                     "speed_attain@1": m["speed_attainment@1"],
                     "found_fastest@1": m["found_fastest@1"],
                     "ndcg": m["ndcg_graded"]})
    return pd.DataFrame(rows)

for name in RNAMES:
    print(f"\n[{name}] by level:")
    print(breakdown(scored[name], "level").round(3).to_string(index=False))

In [ ]:
# --- §5 write metrics.json ---------------------------------------------------
metrics = {
    "params": {"SPEEDUP_STAT": SPEEDUP_STAT, "POOL_RUNS": POOL_RUNS, "TAUS": TAUS,
               "SPEEDUP_CAP": SPEEDUP_CAP, "K_MAX": K_MAX,
               "N_RANDOM_DRAWS": N_RANDOM_DRAWS, "N_BOOTSTRAP": N_BOOTSTRAP,
               "SPEEDUP_LO": SPEEDUP_LO, "SPEEDUP_HI": SPEEDUP_HI,
               "SPEED_QUANT": SPEED_QUANT},
    "eval_table": os.path.abspath(EVAL_TABLE),
    "rerankers": {n: os.path.abspath(p) for n, p in RERANKERS.items()},
    "dataset": {"n_total_kernels": int(n_total), "n_compiled": int(n_comp),
                "compile_rate": n_comp / max(n_total, 1),
                "n_problems": int(n_problems), "n_unsolvable": int(n_unsolvable)},
    "sec2_ranking": {n: {k: (None if pd.isna(v) else float(v))
                         for k, v in sec2[n].items()} for n in RNAMES},
    "sec3_selection": {
        "oracle_correct_ceiling": int(n_solvable_correct),
        "oracle_mean_speedup_capped": float(cap(oracle_speed).mean()),
        "random_mean_speedup_capped": float(rnd_speed_means.mean()),
        "random_mean_correct": float(rnd_ncorrect.mean()),
        "per_reranker": {n: {"mean_picked_speedup_capped": float(cap(rer_speed[n]).mean()),
                             "n_correct_at1": int(rer_correct[n])} for n in RNAMES},
    },
    "sec5_bootstrap_ci": {n: {k: {"point": sec5_ci[n][k][0], "lo": sec5_ci[n][k][1],
                                  "hi": sec5_ci[n][k][2]} for k in BOOT_KEYS}
                          for n in RNAMES},
}
mpath = os.path.join(OUT_DIR, "metrics.json")
with open(mpath, "w") as f:
    json.dump(metrics, f, indent=2)
print(f"wrote {mpath}")
print(f"artifacts in {OUT_DIR}: " + ", ".join(sorted(os.listdir(OUT_DIR))))

### Reading the results
- **§1** bounds everything: `n_unsolvable` and the dataset speed ceiling cap what any reranker
  can achieve. If few problems have an oracle ≥ 1×, don't expect high absolute speedups.
- **§2**: `correctness_pair_acc` / `speed_pair_acc` / `ndcg_graded` near the training-val numbers ⇒
  checkpoint/encoding are sane. `speed_attainment@1` and `found_fastest@1` are the "did we find the
  fastest correct kernel" answer. If pair accuracies sit at 0.5, suspect a wrong checkpoint before
  reading anything else.
- **§3**: each reranker bar should clear the random bar (and its 5–95% band) on both plots; the gap
  to oracle is the remaining headroom.
- **§4**: correct-fast should sit at high within-problem percentiles and compiled-wrong at low ones;
  the speed-discrimination boxplots should rise slow→fast; per-problem AUC mass above 0.5.
- Flip `SPEEDUP_STAT` between `mean`/`min` to check speed conclusions are robust — no re-scoring needed.